# 🦥 Fine-tuning com Unsloth — Classificador de Compras Públicas

**Objetivo:** Treinar um modelo para classificar itens de compras públicas como `SERVIÇO` ou `PRODUTO`, com justificativa e nível de confiança.

**Pipeline:**
```
1. Instalação
2. Dataset via Langfuse
3. Carregamento do modelo base (Mistral 7B Instruct)
4. Configuração do LoRA
5. Treino com SFTTrainer
6. Inferência
7. Export (GGUF / HuggingFace)
```

**Requisitos:** GPU T4 (Colab gratuito), ~15GB VRAM

---
> ⚠️ Vá em `Runtime → Change runtime type → T4 GPU` antes de rodar

## 1. Instalação

In [1]:
%%capture
# Unsloth — instala versão compatível com T4 (CUDA 12.1)
!pip install unsloth
!pip install --upgrade trl transformers accelerate datasets

# Para geração do dataset
!pip install mistralai
!pip install langfuse

In [2]:
# Verifica GPU disponível
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

GPU: Tesla T4
VRAM total: 15.6 GB
CUDA: 12.8


## 2. Dataset via Langfuse

Neste notebook, o dataset e o prompt são consumidos do Langfuse.

1. Execute `python langfuse_seed_assets.py` para enviar os seeds (itens) ao Langfuse
2. Rode a célula abaixo para baixar do Langfuse e gerar `dataset.jsonl`

In [3]:
import json
from pathlib import Path
import os
from langfuse import get_client
from google.colab import userdata # Importa userdata para acessar os segredos

PROMPT_NAME = "classificador-compras-publicas"
PROMPT_LABEL = "production"
DATASET_NAME = "compras-publicas-classificacao"

# Recupera as chaves do Langfuse dos segredos do Colab
LANGFUSE_PUBLIC_KEY = userdata.get('LANGFUSE_PUBLIC_KEY')
LANGFUSE_SECRET_KEY = userdata.get('LANGFUSE_SECRET_KEY')
LANGFUSE_BASE_URL = userdata.get('LANGFUSE_BASE_URL')

# Define as chaves e o base URL como variáveis de ambiente para o Langfuse
os.environ['LANGFUSE_PUBLIC_KEY'] = LANGFUSE_PUBLIC_KEY
os.environ['LANGFUSE_SECRET_KEY'] = LANGFUSE_SECRET_KEY
os.environ['LANGFUSE_BASE_URL'] = LANGFUSE_BASE_URL

langfuse = get_client()


In [8]:
def _safe_get(obj, field, default=None):
    if isinstance(obj, dict):
        return obj.get(field, default)
    return getattr(obj, field, default)


prompt_client = langfuse.get_prompt(PROMPT_NAME)
dataset_client = langfuse.get_dataset(DATASET_NAME)
dataset_items = _safe_get(dataset_client, "items", [])

prompt_config = _safe_get(prompt_client, "config", {}) or {}
PROMPT_MODEL_FROM_LANGFUSE = prompt_config.get("model")

print(f"Prompt recuperado por GET: {PROMPT_NAME}")
print(f"Modelo do prompt (Langfuse config.model): {PROMPT_MODEL_FROM_LANGFUSE}")
print(f"Itens recuperados do dataset: {len(dataset_items)}")


def _render_system_prompt(item_description: str) -> str:
    if hasattr(prompt_client, "compile"):
        return prompt_client.compile(item_descricao=item_description)

    prompt_template = _safe_get(prompt_client, "prompt")
    if not prompt_template:
        raise ValueError("Prompt nao encontrado no Langfuse. Rode o script langfuse_seed_assets.py.")
    if isinstance(prompt_template, list):
        prompt_template = "\n".join(str(p) for p in prompt_template)
    return str(prompt_template).replace("{{item_descricao}}", item_description)


chatml_train = []
chatml_test = []
for item in dataset_items:
    item_input = _safe_get(item, "input", {}) or {}
    item_expected = _safe_get(item, "expected_output", {}) or {}
    item_metadata = _safe_get(item, "metadata", {}) or {}

    item_description = item_input.get("item_descricao", "")
    if not item_description:
        continue

    classification = item_expected.get("classificacao", "PENDENTE")
    confidence = item_expected.get("confianca", "PENDENTE")
    rationale = item_expected.get("justificativa", "Sem justificativa.")

    system_prompt = _render_system_prompt(item_description)

    row = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Classifique o item abaixo:\n\n{item_description}"},
            {
                "role": "assistant",
                "content": (
                    f"CLASSIFICAÇÃO: {classification}\n"
                    f"CONFIANÇA: {confidence}\n"
                    f"JUSTIFICATIVA: {rationale}"
                ),
            },
        ]
    }

    if item_metadata.get("split") == "test":
        chatml_test.append(row)
    else:
        chatml_train.append(row)

train_path = Path("dataset_train.jsonl")
test_path = Path("dataset_test.jsonl")

train_path.write_text(
    "\n".join(json.dumps(row, ensure_ascii=False) for row in chatml_train),
    encoding="utf-8",
)

test_path.write_text(
    "\n".join(json.dumps(row, ensure_ascii=False) for row in chatml_test),
    encoding="utf-8",
)

print(f"dataset_train.jsonl gerado com {len(chatml_train)} exemplos.")
print(f"dataset_test.jsonl gerado com {len(chatml_test)} exemplos.")

Prompt recuperado por GET: classificador-compras-publicas
Modelo do prompt (Langfuse config.model): mistral-small-latest
Itens recuperados do dataset: 40
dataset_train.jsonl gerado com 32 exemplos.
dataset_test.jsonl gerado com 8 exemplos.


## 3. Carregamento do Modelo Base

Usamos **Mistral 7B Instruct** quantizado em 4-bit — cabe no T4 (~5GB VRAM).

In [5]:
from unsloth import FastLanguageModel
import torch

# Opção leve: "unsloth/Llama-3.2-3B-Instruct"  # ~2.5GB em 4-bit
MODEL_NAME = "unsloth/Mistral-7B-Instruct-v0.3"  # padrão deste notebook (~5GB em 4-bit, cabe no T4)
# Alternativas para o T4:
# "unsloth/Llama-3.1-8B-Instruct"    — melhor qualidade, ~5.5GB
# "unsloth/Qwen2.5-7B-Instruct"      — ótimo PT-BR, ~5GB
# "unsloth/Phi-3.5-mini-instruct"    — mais leve, ~3GB

MAX_SEQ_LENGTH = 2048  # cobre bem os itens longos de licitação

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,   # essencial para caber no T4
    dtype=None,          # auto-detecta (float16 no T4)
)

print(f"✅ Modelo carregado: {MODEL_NAME}")
print(f"VRAM usada: {torch.cuda.memory_allocated()/1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


==((====))==  Unsloth 2026.2.1: Fast Mistral patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

✅ Modelo carregado: unsloth/Mistral-7B-Instruct-v0.3
VRAM usada: 4.17 GB


## 4. Configuração do LoRA

**LoRA (Low-Rank Adaptation):** em vez de atualizar todos os ~3B parâmetros, treinamos apenas matrizes pequenas (adaptadores) inseridas nas camadas de atenção. Resultado: treino muito mais rápido e eficiente em memória.

```
Parâmetros totais do modelo:     ~3.000.000.000
Parâmetros treináveis (LoRA r=16):  ~20.000.000  (~0.7%)
```

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    # --- LoRA rank: controla capacidade dos adaptadores ---
    # r=8  → menor, mais rápido, menos expressivo
    # r=16 → bom equilíbrio (recomendado)
    # r=32 → mais expressivo, mais memória
    r=16,

    # Módulos onde os adaptadores são inseridos
    # Para classificação, focar em atenção + FFN é suficiente
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # atenção
        "gate_proj", "up_proj", "down_proj",       # FFN
    ],

    lora_alpha=16,   # escala dos adaptadores (manter = r é boa prática)
    lora_dropout=0,  # 0 funciona bem com Unsloth
    bias="none",

    # gradient checkpointing da Unsloth — economiza ~40% de VRAM no T4
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Mostra quantos parâmetros são treináveis
total = sum(p.numel() for p in model.parameters())
treinavel = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parâmetros totais:     {total:,}")
print(f"Parâmetros treináveis: {treinavel:,} ({treinavel/total*100:.2f}%)")

Unsloth 2026.2.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Parâmetros totais:     3,800,305,664
Parâmetros treináveis: 41,943,040 (1.10%)


## 5. Preparação do Dataset

In [9]:
from datasets import Dataset
import json

# Carrega apenas o split de treino gerado na etapa 2
with open("dataset_train.jsonl", encoding="utf-8") as f:
    training_examples = [json.loads(line) for line in f if line.strip()]

print(f"Exemplos de treino carregados: {len(training_examples)}")

def format_for_training(example):
    """
    Converte o formato messages para texto usando o chat template do modelo.
    O modelo aprende a completar a resposta do assistente dado user+system.
    """
    formatted_text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False  # False = inclui a resposta do assistente
    )
    return {"text": formatted_text}

dataset_hf = Dataset.from_list(training_examples).map(format_for_training)

# Visualiza como o texto fica formatado para o modelo
print("\n--- Exemplo formatado (primeiros 500 chars) ---")
print(dataset_hf[0]["text"][:500])
print("...")

# Estatísticas de tamanho dos exemplos
import numpy as np
token_lengths = [len(tokenizer.encode(example["text"])) for example in dataset_hf]
print(f"\nTokens por exemplo: min={min(token_lengths)} | média={np.mean(token_lengths):.0f} | max={max(token_lengths)}")

Exemplos de treino carregados: 32


Map:   0%|          | 0/32 [00:00<?, ? examples/s]


--- Exemplo formatado (primeiros 500 chars) ---
<s>[INST] Classifique o item abaixo:

Aquisição de 90 nobreaks 1200VA para proteção de equipamentos críticos.[/INST] CLASSIFICAÇÃO: PRODUTO
CONFIANÇA: Alta
JUSTIFICATIVA: Item específico e tangível, claramente identificado como produto.</s>
...

Tokens por exemplo: min=83 | média=100 | max=128


## 6. Treino

> ⏱️ Tempo estimado para 300 exemplos × 3 épocas no T4: ~15-20 minutos

In [10]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset_hf,
    args=SFTConfig(
        output_dir="./checkpoints",

        # --- Épocas e batch ---
        num_train_epochs=3,              # 3 épocas é bom para datasets pequenos
        per_device_train_batch_size=2,   # T4 aguenta 2 com seq_len=2048
        gradient_accumulation_steps=4,  # batch efetivo = 2*4 = 8

        # --- Learning rate ---
        learning_rate=2e-4,             # padrão recomendado para LoRA
        warmup_steps=10,
        lr_scheduler_type="cosine",

        # --- Precisão: T4 usa fp16 (não tem bf16) ---
        fp16=True,
        bf16=False,

        # --- Logging e salvamento ---
        logging_steps=10,
        save_strategy="epoch",
        report_to="none",  # desativa wandb

        # --- Otimizador eficiente em memória ---
        optim="adamw_8bit",

        # --- Dataset ---
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_num_proc=2,
    ),
)

print("Iniciando treino...")
print(f"Steps totais: {trainer.args.max_steps if trainer.args.max_steps > 0 else 'calculado por épocas'}")

training_result = trainer.train()

print(f"\n✅ Treino concluído!")
print(f"Loss final: {training_result.training_loss:.4f}")
print(f"Tempo total: {training_result.metrics['train_runtime']:.0f}s")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/32 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Iniciando treino...
Steps totais: calculado por épocas


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 32 | Num Epochs = 3 | Total steps = 12
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)


Step,Training Loss
10,1.872110



✅ Treino concluído!
Loss final: 1.7067
Tempo total: 67s


## 7. Inferência

Testa o modelo treinado antes de exportar.

In [11]:
import logging
import warnings

# Silencia avisos de depreciação do Transformers e erros de formatação de log
logging.getLogger("transformers.modeling_attn_mask_utils").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

print("✅ Avisos silenciados. As próximas execuções devem ter uma saída limpa.")

✅ Avisos silenciados. As próximas execuções devem ter uma saída limpa.


In [12]:
from unsloth import FastLanguageModel
import torch
from langfuse import get_client

# Ativa modo de inferência otimizado
FastLanguageModel.for_inference(model)

PROMPT_NAME = "classificador-compras-publicas"
PROMPT_LABEL = "production"

_langfuse = get_client()
_prompt_client = _langfuse.get_prompt(PROMPT_NAME, label=PROMPT_LABEL)


def _render_system_prompt_inference(item_description: str) -> str:
    if hasattr(_prompt_client, "compile"):
        return _prompt_client.compile(item_descricao=item_description)

    prompt_template = getattr(_prompt_client, "prompt", None)
    if not prompt_template:
        raise ValueError("Prompt nao encontrado no Langfuse para inferencia.")
    if isinstance(prompt_template, list):
        prompt_template = "\n".join(str(p) for p in prompt_template)
    return str(prompt_template).replace("{{item_descricao}}", item_description)


def classify_item(item_description: str, verbose: bool = True) -> str:
    messages = [
        {"role": "system", "content": _render_system_prompt_inference(item_description)},
        {"role": "user", "content": f"Classifique o item abaixo:\n\n{item_description}"},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    input_ids = inputs["input_ids"]

    with torch.no_grad():
        output = model.generate(
            input_ids=input_ids,
            attention_mask=inputs.get("attention_mask"),
            max_new_tokens=150,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    answer = tokenizer.decode(
        output[0][input_ids.shape[1]:],
        skip_special_tokens=True,
    ).strip()

    if verbose:
        print(f"INPUT: {item_description[:80]}...")
        print(f"OUTPUT:\n{answer}")
        print("-" * 60)

    return answer


def _field(obj, key, default=None):
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)

test_cases = [
    "Contratação de empresa especializada para realizar o evento 75ª Reunião Ordinária do CONAPREV, nos dias 23 a 24 de março de 2023, em Goiânia/GO, com carga horária de 16 horas.",
    "Aquisição de 200 cadeiras ergonômicas para escritório, modelo executivo, com regulagem de altura e apoio lombar ajustável.",
    "Prestação de serviços de desenvolvimento de software para modernização do sistema de gestão de contratos.",
    "Fornecimento de 10 ar-condicionados split 12.000 BTUs, inverter, instalação inclusa.",
    "Iphone 11 pro R$ 2769,00 apple smarthwatch",
]

for test_case in test_cases:
    classify_item(test_case)

INPUT: Contratação de empresa especializada para realizar o evento 75ª Reunião Ordinári...
OUTPUT:
CLASSIFICAÇÃO: SERVIÇO
CONFIANÇA: Alta
JUSTIFICATIVA: O item descreve a contratação de uma empresa para realizar um evento, o que é claramente um serviço.
------------------------------------------------------------
INPUT: Aquisição de 200 cadeiras ergonômicas para escritório, modelo executivo, com reg...
OUTPUT:
CLASSIFICAÇÃO: PRODUTO
CONFIANÇA: Alta
JUSTIFICATIVA: O item descreve a aquisição de 200 cadeiras, que são itens tangíveis e padronizados, caracterizando-os como produtos.
------------------------------------------------------------
INPUT: Prestação de serviços de desenvolvimento de software para modernização do sistem...
OUTPUT:
CLASSIFICAÇÃO: SERVIÇO
CONFIANÇA: Alta
JUSTIFICATIVA: O item descreve a prestação de serviços de desenvolvimento de software, que é claramente um serviço.
------------------------------------------------------------
INPUT: Fornecimento de 10 ar-condicion

## 8. Avaliação

Calcula métricas básicas num conjunto de teste separado.

In [13]:
def _field(obj, key, default=None):
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def _normalize_label(value: str) -> str:
    normalized_text = (value or "").upper().strip()
    if normalized_text in {"SERVICO", "SERVIÇO"}:
        return "SERVIÇO"
    if normalized_text == "PRODUTO":
        return "PRODUTO"
    return ""


test_items = []
for item in dataset_items:
    metadata = _field(item, "metadata", {}) or {}
    if metadata.get("split") != "test":
        continue

    item_input = _field(item, "input", {}) or {}
    item_expected = _field(item, "expected_output", {}) or {}

    item_description = (item_input.get("item_descricao") or "").strip()
    expected_label = _normalize_label(item_expected.get("classificacao", ""))

    if item_description and expected_label:
        test_items.append({"item_description": item_description, "expected_label": expected_label})

num_test = len(test_items)
if num_test == 0:
    raise ValueError("Nenhum item de teste válido encontrado (metadata.split='test').")

print(f"Avaliando em {num_test} exemplos de teste do Langfuse...\n")

correct_predictions = 0
evaluation_results = []

for example in test_items:
    item_text = example["item_description"]
    expected_label = example["expected_label"]

    model_answer = classify_item(item_text, verbose=False)
    predicted_label = "SERVIÇO" if "SERVIÇO" in model_answer else "PRODUTO"

    is_correct = expected_label == predicted_label
    correct_predictions += is_correct
    evaluation_results.append({"expected_label": expected_label, "predicted_label": predicted_label, "is_correct": is_correct})
    print(f"{'✅' if is_correct else '❌'} Esperado: {expected_label} | Previsto: {predicted_label}")

accuracy = correct_predictions / num_test * 100
print(f"\n📊 Acurácia: {correct_predictions}/{num_test} = {accuracy:.1f}%")

Avaliando em 8 exemplos de teste do Langfuse...

✅ Esperado: SERVIÇO | Previsto: SERVIÇO
✅ Esperado: PRODUTO | Previsto: PRODUTO
✅ Esperado: SERVIÇO | Previsto: SERVIÇO
✅ Esperado: PRODUTO | Previsto: PRODUTO
✅ Esperado: PRODUTO | Previsto: PRODUTO
✅ Esperado: SERVIÇO | Previsto: SERVIÇO
✅ Esperado: PRODUTO | Previsto: PRODUTO
✅ Esperado: PRODUTO | Previsto: PRODUTO

📊 Acurácia: 8/8 = 100.0%


## 9. Export

Escolha uma das opções abaixo dependendo de como quer usar o modelo.

In [ ]:
# ============================================================
# Opção A — Salva apenas os adaptadores LoRA (~100MB)
# Útil para: voltar a treinar depois, ou carregar com Unsloth
# ============================================================

model.save_pretrained("modelo-lora")
tokenizer.save_pretrained("modelo-lora")
print("✅ Adaptadores LoRA salvos em ./modelo-lora")

In [ ]:
# ============================================================
# Opção B — Exporta para GGUF (rodar localmente com Ollama)
# quantization_method:
#   q4_k_m  → melhor equilíbrio tamanho/qualidade (recomendado)
#   q5_k_m  → um pouco maior, melhor qualidade
#   q8_0    → quase sem perda de qualidade, maior
# ============================================================

model.save_pretrained_gguf("modelo-gguf", tokenizer, quantization_method="q4_k_m")
print("✅ GGUF salvo — para usar com Ollama:")
print("   ollama create classificador -f ./modelo-gguf/Modelfile")
print("   ollama run classificador")

In [ ]:
# ============================================================
# Opção C — Sobe para HuggingFace Hub
# ============================================================

# HF_TOKEN = "hf_..."  # seu token do HuggingFace
# HF_REPO  = "seu-usuario/classificador-compras-publicas"

# model.push_to_hub(HF_REPO, token=HF_TOKEN)
# tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
# print(f"✅ Modelo no HuggingFace: https://huggingface.co/{HF_REPO}")

print("Descomente e preencha HF_TOKEN e HF_REPO para subir ao HuggingFace Hub")

## 10. Dicas para Melhorar

### Se a acurácia ficou abaixo de 90%:
- Aumente o dataset (tente 500+ exemplos)
- Aumente `num_train_epochs` para 5
- Verifique se o dataset está balanceado (50/50 serviço/produto)

### Se o modelo não segue o formato de saída:
- Garanta que o prompt ativo no Langfuse esteja consistente para treino/inferência
- Reduza `temperature` para 0.05 na inferência
- Adicione mais exemplos com o formato exato desejado

### Para expandir para mais classes (ex: Obra, Locação):
- Atualize o prompt no Langfuse e os rótulos do dataset
- Mantenha o dataset balanceado entre todas as classes
- Aumente `r=32` no LoRA para mais capacidade

### Referências:
- [Unsloth GitHub](https://github.com/unslothai/unsloth)
- [Documentação Unsloth](https://docs.unsloth.ai)
- [TRL SFTTrainer](https://huggingface.co/docs/trl/sft_trainer)